<a href="https://colab.research.google.com/github/Divyansh-yecho/Lunar_lander_toy_model/blob/main/lunar_lander_(final).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Lunar Lander — DQN with Stable-Baselines3

Training a Deep Q-Network (DQN) agent to solve the `LunarLander-v3` environment from Gymnasium.

**Highlights:**
- DQN with experience replay and target network
- Checkpoint saving every 10k steps
- Training logs written to file (not cluttering the notebook)
- TensorBoard integration for live training curves
- Video recording of the final agent

## 1. Install Dependencies

In [ ]:
!pip install -q gymnasium[box2d] stable-baselines3 tensorboard tbparse

## 2. Imports

In [ ]:
import os
import logging

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback, CallbackList
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from gymnasium.wrappers import RecordVideo

## 3. Mount Google Drive & Set Up Directories

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR       = "/content/drive/MyDrive/lunar_lander_project"
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
MODEL_DIR      = os.path.join(BASE_DIR, "models")
LOG_DIR        = os.path.join(BASE_DIR, "logs")
TB_LOG_DIR     = os.path.join(BASE_DIR, "tensorboard_logs")   # TensorBoard logs
VIDEO_DIR      = os.path.join(BASE_DIR, "videos")

for d in [CHECKPOINT_DIR, MODEL_DIR, LOG_DIR, TB_LOG_DIR, VIDEO_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Directories ready")
print(f"   TensorBoard logs → {TB_LOG_DIR}")

## 4. Configure File Logging

All verbose SB3 output is redirected to `training.log` so the notebook stays clean.

In [ ]:
log_file = os.path.join(LOG_DIR, "training.log")

logging.basicConfig(
    filename=log_file,
    filemode="a",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

sb3_logger = logging.getLogger("stable_baselines3")
sb3_logger.setLevel(logging.INFO)
sb3_logger.propagate = True

print(f"📝 File log → {log_file}")

## 5. Create Environments

In [ ]:
env      = Monitor(gym.make("LunarLander-v3"), filename=os.path.join(LOG_DIR, "monitor"))
eval_env = Monitor(gym.make("LunarLander-v3"))   # separate env for EvalCallback

print(f"Observation space : {env.observation_space}")
print(f"Action space      : {env.action_space}")

## 6. Build DQN Model

`tensorboard_log` points to our dedicated TensorBoard directory.  
SB3 creates a `DQN_phase1/`, `DQN_phase2/`, `DQN_phase3/` run folder inside it — one per training phase.

In [ ]:
model = DQN(
    "MlpPolicy",
    env,
    learning_rate=5e-4,
    buffer_size=100_000,
    learning_starts=5_000,
    batch_size=64,
    gamma=0.99,
    train_freq=4,
    target_update_interval=1_000,
    exploration_fraction=0.15,
    exploration_final_eps=0.02,
    verbose=0,                    # suppressed — goes to log file
    tensorboard_log=TB_LOG_DIR,   # TensorBoard enabled
)

print("✅ Model ready")

## 7. Set Up Callbacks

- **CheckpointCallback** — saves model every 10k steps to `checkpoints/`
- **EvalCallback** — evaluates every 10k steps and writes `eval/mean_reward` to TensorBoard automatically

In [ ]:
checkpoint_cb = CheckpointCallback(
    save_freq=10_000,
    save_path=CHECKPOINT_DIR,
    name_prefix="dqn_lunar",
)

eval_cb = EvalCallback(
    eval_env,
    best_model_save_path=MODEL_DIR,
    log_path=LOG_DIR,
    eval_freq=10_000,
    n_eval_episodes=10,
    deterministic=True,
    verbose=0,
)

callbacks = CallbackList([checkpoint_cb, eval_cb])
print("✅ Callbacks ready")

## 8. Launch TensorBoard

Run this cell **before or during training** to watch live curves update in real time.  
Key metrics to watch: `eval/mean_reward`, `train/loss`, `train/exploration_rate`

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {TB_LOG_DIR}

## 9. Train — Phase 1 (200k steps)

In [ ]:
print("🚀 Training phase 1 …")
model.learn(
    total_timesteps=200_000,
    callback=callbacks,
    progress_bar=True,
    tb_log_name="DQN_phase1",
)

mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=10)
logging.info(f"Phase 1 eval — mean: {mean_r:.2f}, std: {std_r:.2f}")
print(f"Phase 1 → mean reward: {mean_r:.2f} ± {std_r:.2f}")

## 10. Continue Training — Phase 2 (300k steps)

In [ ]:
ckpt_path = os.path.join(CHECKPOINT_DIR, "dqn_lunar_200000_steps")
model = DQN.load(ckpt_path, env=env, tensorboard_log=TB_LOG_DIR)

print("🚀 Training phase 2 …")
model.learn(
    total_timesteps=300_000,
    callback=callbacks,
    progress_bar=True,
    tb_log_name="DQN_phase2",
)

mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=10)
logging.info(f"Phase 2 eval — mean: {mean_r:.2f}, std: {std_r:.2f}")
print(f"Phase 2 → mean reward: {mean_r:.2f} ± {std_r:.2f}")

## 11. Continue Training — Phase 3 (300k steps)

In [ ]:
print("🚀 Training phase 3 …")
model.learn(
    total_timesteps=300_000,
    callback=callbacks,
    progress_bar=True,
    tb_log_name="DQN_phase3",
)

mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=20)
logging.info(f"Phase 3 final eval — mean: {mean_r:.2f}, std: {std_r:.2f}")
print(f"Final → mean reward: {mean_r:.2f} ± {std_r:.2f}")

model.save(os.path.join(MODEL_DIR, "final_model"))
print("💾 Final model saved")

## 12. Plot Training Curves from TensorBoard Data

Reads the TensorBoard event files and plots reward + loss curves directly in the notebook.

In [ ]:
from tbparse import SummaryReader

reader = SummaryReader(TB_LOG_DIR, pivot=False)
df = reader.scalars

print("Available tags:", df['tag'].unique().tolist())

eval_df  = df[df['tag'] == 'eval/mean_reward'].sort_values('step')
train_df = df[df['tag'] == 'train/loss'].sort_values('step')
eps_df   = df[df['tag'] == 'rollout/exploration_rate'].sort_values('step')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Eval reward
axes[0].plot(eval_df['step'], eval_df['value'], color='royalblue', linewidth=2)
axes[0].axhline(y=200, color='green', linestyle='--', alpha=0.6, label='Solved (200)')
axes[0].set_title('Eval Mean Reward')
axes[0].set_xlabel('Timesteps')
axes[0].set_ylabel('Mean Reward')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Training loss
if not train_df.empty:
    axes[1].plot(train_df['step'], train_df['value'], color='tomato', linewidth=1, alpha=0.8)
    axes[1].set_title('Training Loss')
    axes[1].set_xlabel('Timesteps')
    axes[1].set_ylabel('Loss')
    axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'No loss data', ha='center', va='center', transform=axes[1].transAxes)

# Exploration rate
if not eps_df.empty:
    axes[2].plot(eps_df['step'], eps_df['value'], color='darkorange', linewidth=2)
    axes[2].set_title('Exploration Rate (ε)')
    axes[2].set_xlabel('Timesteps')
    axes[2].set_ylabel('Epsilon')
    axes[2].grid(alpha=0.3)
else:
    axes[2].text(0.5, 0.5, 'No epsilon data', ha='center', va='center', transform=axes[2].transAxes)

plt.suptitle('DQN LunarLander-v3 — Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "training_curves.png"), dpi=150, bbox_inches='tight')
plt.show()
print("📊 Plot saved → training_curves.png")

## 13. Record Agent Videos

In [ ]:
video_env = gym.make("LunarLander-v3", render_mode="rgb_array")
video_env = RecordVideo(
    video_env,
    video_folder=VIDEO_DIR,
    episode_trigger=lambda e: True,
    name_prefix="lunar_agent",
)

for episode in range(2):
    obs, _ = video_env.reset()
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, truncated, _ = video_env.step(action)
        if done or truncated:
            break

video_env.close()
print(f"🎬 Videos saved to {VIDEO_DIR}")
print(os.listdir(VIDEO_DIR))

## 14. Playback Videos in Notebook

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode

def show_video(path):
    mp4 = open(path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'''
        <video width="400" controls>
            <source src="{data_url}" type="video/mp4">
        </video>
    '''))

video_files = sorted([f for f in os.listdir(VIDEO_DIR) if f.endswith(".mp4")])
for vf in video_files[:2]:
    print(f"▶ {vf}")
    show_video(os.path.join(VIDEO_DIR, vf))

## 15. View Training Log (last 20 lines)

In [ ]:
with open(log_file) as f:
    lines = f.readlines()

print(f"Total log lines: {len(lines)}")
print("\n--- Last 20 entries ---")
print("".join(lines[-20:]))